In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from src.features import make_features

FEATS = ["hour", "weekday", "lag24_n", "lag168_n", "temp"]
UNSEEN_SITES = ["Panther", "Swan"]
MAX_MISSING = 0.20          # the same quality rule as notebook 01

df_clean = pd.read_parquet("../data/processed/electricity_clean.parquet")
temp = pd.read_parquet("../data/processed/air_temperature.parquet")
train_buildings = pd.read_csv("../configs/train_buildings.csv")["building"].tolist()
print("df_clean:", df_clean.shape, "index type:", type(df_clean.index).__name__)

# Buildings of the unseen sites: same rule (at most 20% missing), checked on 2017 only
candidates = [b for b in df_clean.columns if b.split("_")[0] in UNSEEN_SITES]
missing_2017 = df_clean.loc[df_clean.index.year == 2017, candidates].isna().mean()
unseen_buildings = missing_2017[missing_2017 <= MAX_MISSING].index.tolist()
print(len(candidates), "buildings on Panther/Swan,", len(unseen_buildings), "pass the 20% rule in 2017")
print(pd.Series([b.split("_")[0] for b in unseen_buildings]).value_counts().to_string())


def build(buildings):
    parts = []
    for b in buildings:
        f = make_features(b, df_clean, temp)
        num = f.columns.drop("building")
        f[num] = f[num].astype("float32")
        parts.append(f)
    out = pd.concat(parts)
    out["building"] = out["building"].astype("category")
    return out


data = build(train_buildings)                      # the 1,083 training buildings, 2016 + 2017
unseen = build(unseen_buildings)
unseen = unseen[unseen.index.year == 2017]         # unseen sites: 2017 only, never trained on
print(len(data), "rows for known buildings,", len(unseen), "rows for unseen buildings")
print("temperature range:", float(data["temp"].min()), "to", float(data["temp"].max()), "(should be °C)")

df_clean: (17544, 1578) index type: DatetimeIndex
124 buildings on Panther/Swan, 124 pass the 20% rule in 2017
Panther    105
Swan        19
19000152 rows for known buildings, 1086240 rows for unseen buildings
temperature range: -28.899999618530273 to 48.29999923706055 (should be °C)


In [ ]:
train = data[data.index.year == 2016].dropna(subset=["load_n"])
gbm = HistGradientBoostingRegressor(random_state=0).fit(train[FEATS], train["load_n"])
del train


def score(df, group):
    """Per-building MAE (kWh) of the model and of both baselines, on the same rows."""
    d = df.dropna(subset=["load", "lag24", "lag168", "scale"])
    pred = gbm.predict(d[FEATS]) * d["scale"].to_numpy()
    load = d["load"].to_numpy()
    err = pd.DataFrame({
        "building": d["building"].astype(str).to_numpy(),
        "mae_model": np.abs(load - pred),
        "mae_yesterday": np.abs(load - d["lag24"].to_numpy()),
        "mae_lastweek": np.abs(load - d["lag168"].to_numpy()),
        "mean_load": load,
    })
    res = err.groupby("building").mean()
    res["best_baseline"] = res[["mae_yesterday", "mae_lastweek"]].min(axis=1)
    res["wins"] = res["mae_model"] < res["best_baseline"]
    res["improvement"] = 1 - res["mae_model"] / res["best_baseline"]
    res["rel_mae"] = res["mae_model"] / res["mean_load"]
    res["site"] = res.index.str.split("_").str[0]
    res["group"] = group
    return res


known = score(data[data.index.year == 2017], "known sites")
new = score(unseen, "unseen sites")
new["group"] = "unseen: " + new["site"]
both = pd.concat([known, new])

summary = both.groupby("group").agg(
    buildings=("wins", "size"),
    win_pct=("wins", lambda s: 100 * s.mean()),
    median_improvement_pct=("improvement", lambda s: 100 * s.median()),
    median_rel_mae=("rel_mae", "median"))
summary.loc["unseen: both sites"] = [len(new), 100 * new["wins"].mean(),
                                     100 * new["improvement"].median(), new["rel_mae"].median()]
summary["buildings"] = summary["buildings"].astype(int)
print(summary.round(3).to_string())

known.to_csv("../results/tables/cv_known_2017.csv")
new.to_csv("../results/tables/cv_unseen_sites.csv")
summary.to_csv("../results/tables/cv_unseen_summary.csv")

order = ["known sites"] + sorted(new["group"].unique())
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([100 * both.loc[both["group"] == g, "improvement"].clip(-1, 1) for g in order],
           showfliers=False)
ax.set_xticks(range(1, len(order) + 1), order)
ax.axhline(0, color="grey", lw=1, ls="--")
ax.set_ylabel("improvement over the stronger baseline (%)")
ax.set_title("Gradient boosting trained on 2016, known sites; tested on 2017")
plt.tight_layout()
plt.savefig("../results/figures/cv_unseen_sites.png", dpi=150)
plt.show()